the architecture of the visual cortex

they showed that many neurones in visual cortex react only to image,and then they can overlap,and showed also that there are neurones that react only to horizontal line and one's for vertical line

In [1]:
import numpy as np
import torch
from sklearn.datasets import load_sample_images
sample_images = np.stack(load_sample_images()["images"])
sample_images = torch.tensor(sample_images, dtype=torch.float32) / 255

: 

In [2]:
sample_images.shape

torch.Size([2, 427, 640, 3])

: 

: 

In [3]:
sample_images_permuted = sample_images.permute(0, 3, 1, 2)

: 

: 

In [4]:
import torchvision
import torchvision.transforms.v2 as T
cropped_images = T.CenterCrop((70, 120))(sample_images_permuted)
cropped_images.shape

torch.Size([2, 3, 70, 120])

: 

: 

In [5]:
import torch.nn as nn 
torch.manual_seed(42)
conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7)
fmaps = conv_layer(cropped_images)
fmaps.shape

torch.Size([2, 32, 64, 114])

: 

: 

we lose 6 pixels in height and width because we dont activation padding and for that we crop some pixels to apply convulution

In [6]:
conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7,padding="same")
fmaps = conv_layer(cropped_images)
fmaps.shape

torch.Size([2, 32, 70, 120])

: 

: 

In [7]:
max_pool = nn.MaxPool2d(kernel_size=2)

: 

: 

In [8]:
import torch.nn.functional as F
class DepthPool(torch.nn.Module):
    def __init__(self, kernel_size, stride=None, padding=0):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride if stride is not None else kernel_size
        self.padding = padding
    def forward(self, inputs):
        batch, channels, height, width = inputs.shape
        Z = inputs.view(batch, channels, height * width) # merge spatial dims
        Z = Z.permute(0, 2, 1) # switch spatial and channels dims
        Z = F.max_pool1d(Z, kernel_size=self.kernel_size,
        stride=self.stride,
        padding=self.padding) # compute max pool
        Z = Z.permute(0, 2, 1) # switch back spatial and channels dims
        return Z.view(batch, -1, height, width) # unmerge spatial dims

: 

: 

In [9]:
global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1)
output = global_avg_pool(cropped_images)

: 

: 

In [10]:
output

tensor([[[[0.6434]],

         [[0.5972]],

         [[0.5825]]],


        [[[0.7631]],

         [[0.2601]],

         [[0.1085]]]])

: 

: 

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


: 

: 

In [12]:
from functools import partial
DefaultConv2d = partial(nn.Conv2d, kernel_size=3, padding="same")
model = nn.Sequential(
DefaultConv2d(in_channels=1, out_channels=64, kernel_size=7),
nn.ReLU(),
nn.MaxPool2d(kernel_size=2),
DefaultConv2d(in_channels=64, out_channels=128), nn.ReLU(),
DefaultConv2d(in_channels=128, out_channels=128), nn.ReLU(),
nn.MaxPool2d(kernel_size=2),
DefaultConv2d(in_channels=128, out_channels=256), nn.ReLU(),
DefaultConv2d(in_channels=256, out_channels=256), nn.ReLU(),
nn.MaxPool2d(kernel_size=2),
nn.Flatten(),
nn.Linear(in_features=2304, out_features=128), nn.ReLU(),
nn.Dropout(0.5),
nn.Linear(in_features=128, out_features=64), nn.ReLU(),
nn.Dropout(0.5),
nn.Linear(in_features=64, out_features=10),
).to(device)

: 

: 

In [ ]:
import torch.nn as nn
class ResidualUnit(nn.Module):
    def __init__(self,in_channels,out_channels,stride=1):
        super().__init__()
        DefaultConv2d = partial(nn.Conv2d,kernel_size=3,padding=1,stride=1,bias=False)
        self.main_layers = nn.Sequential(
            DefaultConv2d(in_channels,out_channels,stride=stride)
            ,nn.BatchNorm2d(out_channels),
            nn.ReLU,
            DefaultConv2d(in_channels,out_channels),
            nn.BatchNorm2d(out_channels)
        )
        if stride >1 :
            self.skip_connection = nn.Sequential(
                                    DefaultConv2d(in_channels, out_channels, kernel_size=1,
                                    stride=stride, padding=0),
                                    nn.BatchNorm2d(out_channels),
                                    )
        else:
            self.skip_connection = nn.Identity()
        def forward(self,inputs):
            return F.relu(self.main_layers(inputs) + self.skip_connection(inputs))

: 

: 

In [ ]:
class ResNet34(nn.Module):
    def __init__(self):
        super().__init__()
        layers = [
        nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7,stride=2,
        padding=3, bias=False),
        nn.BatchNorm2d(num_features=64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        ]
        prev_filters = 64
        for filters in [64] * 3 + [128] * 4 + [256] * 6 + [512] * 3:
            stride = 1 if filters == prev_filters else 2
            layers.append(ResidualUnit(prev_filters, filters,
            stride=stride))
            prev_filters = filters
            layers += [
            nn.AdaptiveAvgPool2d(output_size=1),
            nn.Flatten(),
            nn.LazyLinear(10),
            ]
        self.resnet = nn.Sequential(*layers)
    def forward(self, inputs):
        return self.resnet(inputs)

: 

: 

we know that deepper models learn better than the simpler one but we have degredation problem that lead the deeper models to overfitting and simpler learn something not that deep but common,but the real problem the models have vanishing gradient ,so we add the input to it to solve it ,if the deeper models converge aprox to zero we still the have the input f(x) + x

In [ ]:
weights = torchvision.models.ConvNeXt_Base_Weights.IMAGENET1K_V1
model = torchvision.models.convnext_base(weights=weights).to(device)

In [ ]:
DefaultFlowers102 = partial(torchvision.datasets.Flowers102,
root="datasets",
transform=weights.transforms(),
download=True)
train_set = DefaultFlowers102(split="train")
valid_set = DefaultFlowers102(split="val")
test_set = DefaultFlowers102(split="test")

In [ ]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=32)
test_loader = DataLoader(test_set, batch_size=32)

In [ ]:
n_classes = 102 # len(class_names) == 102
model.classifier[2] = nn.Linear(1024, n_classes).to(device)

In [ ]:
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

In [ ]:
import torchvision.transforms.v2 as T
transforms = T.Compose([
T.RandomHorizontalFlip(p=0.5),
T.RandomRotation(degrees=30),
T.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0)),
T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2,
hue=0.1),
T.ToImage(),
T.ToDtype(torch.float32, scale=True),
T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])